In [1]:
import cv2
import os
import sys
import shutil
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

In [2]:
# Me aseguro de que el directorio raíz del proyecto esté en el sys.path
project_root = Path(os.path.abspath("")).parent  

# Añado el directorio raíz al sys.path si no está ya presente
if project_root not in sys.path:
    sys.path.append(str(project_root))

In [3]:
# Importo las funciones de configuración
from src.config import interim_data_dir, processed_data_dir, load_config

Current working directory: /home/jorge/development/WasteImageReconstructionDL/notebooks
Loading configuration from /home/jorge/development/WasteImageReconstructionDL/src/config.yaml


In [4]:
# Cargo la configuración 
config = load_config()

Loading configuration from /home/jorge/development/WasteImageReconstructionDL/src/config.yaml


### Procesamiento de imagenes: redimensionado y escalado

In [5]:
# Funciones de redimensionado y escalado de imágenes
def redimensionar_imagen(img, target_size):
    h, w = img.shape[:2]
    target_h, target_w = target_size
    aspect_ratio = w / h
    target_aspect_ratio = target_w / target_h

    if aspect_ratio > target_aspect_ratio:
        new_w = target_w
        new_h = int(np.round(new_w / aspect_ratio))
    else:
        new_h = target_h
        new_w = int(np.round(new_h * aspect_ratio))

    img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return img_resized

def escalar_imagen(img_resized, target_size):
    target_h, target_w = target_size
    new_h, new_w = img_resized.shape[:2]

    img_scaled = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    y_offset = max((target_h - new_h) // 2, 0)
    x_offset = max((target_w - new_w) // 2, 0)

    img_scaled[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = img_resized

    return img_scaled

def process_image(img_path, output_subdir, target_size=(256, 256)):
    img = cv2.imread(img_path)
    img_resized = redimensionar_imagen(img, target_size)
    #img_scaled = escalar_imagen(img_resized, target_size)
    img_name = os.path.basename(img_path)
    img_scaled_path = os.path.join(output_subdir, img_name)
    #cv2.imwrite(img_scaled_path, img_scaled)
    cv2.imwrite(img_scaled_path, img_resized)

In [6]:
# Directorios de entrada y salida
image_path = interim_data_dir() / 'dataset_png'
output_path_scaled = interim_data_dir() / 'dataset_scaled_2'
os.makedirs(output_path_scaled, exist_ok=True)

In [7]:
# Obtener todas las rutas de imágenes y subdirectorios correspondientes
tasks = []
for subdir, _, files in os.walk(image_path):
    for file in files:
        if file.endswith(".png"):
            img_path = os.path.join(subdir, file)
            relative_subdir = os.path.relpath(subdir, image_path)
            output_subdir = os.path.join(output_path_scaled, relative_subdir)
            if not os.path.exists(output_subdir):
                os.makedirs(output_subdir)
            tasks.append((img_path, output_subdir))

In [8]:
# Procesar las imágenes en paralelo
max_workers = None  # None para usar todos los hilos disponibles
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    executor.map(lambda p: process_image(*p), tasks)


In [9]:
print(f"Las imágenes redimensionadas y escaladas se han guardado en {output_path_scaled}")


Las imágenes redimensionadas y escaladas se han guardado en /home/jorge/development/WasteImageReconstructionDL/data/interim/dataset_scaled_2
